# Install software (on CPU)

Now, we need to reinstall the software on the CPU. This is again done by cloning the repo and installing the required software. This time, do **not** active the GPU! This will take a while and you will be asked to restart the session. After the restart, the software is installed.

In [ ]:
%cd /content
!apt-get install git-lfs
!git lfs install
!git clone https://github.com/StringJAX/towards_type_IIB_foundation_model.git
%cd towards_type_IIB_foundation_model
%pip install -r requirements.txt
%cd transformer

# Evaluate on CPU

Now we need to simply load the previously generated samples and evaluate them. This will again access your drive to load the samples from there. Make sure to change things accordingly if you have changed anything in the sampling script (e.g. the file name).

## 1. Evaluate samples from the pre-trained and GRPO refined model

Here we will evaluate the samples from the model that has been pre-trained on all 20 geometries and further refined with GRPO. Make sure to change any of the arguments accordingly, if you have changed them in the sample notebook. 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAMPLE_DIR = '/content/drive/MyDrive/IIB_foundation_model_samples'
import os; os.makedirs(SAMPLE_DIR, exist_ok=True)
%cd /content/towards_type_IIB_foundation_model/transformer
from evaluate import evaluate
import warnings
warnings.filterwarnings("ignore", module=r"^jaxvacua(\.|$)")

file_name="all_h12_sample"
samples_path = f'{SAMPLE_DIR}/{file_name}.p'
samples = evaluate(RUN_ID=37, load_ckpt="GRPO_2/checkpoint_200", num_samples=100, only_evaluate=True, samples_file=samples_path, single=True)

## 2. Evaluate samples from the fine-tuned model

Here we will evaluate the samples from the model that has been pre-trained on all geometries with $h^{1,2}\leq4$ and then fine-tuned on a small dataset with $h^{1,2}=5$.

In [ ]:
file_name="h12_5_finetune_sample"
samples_path = f'{SAMPLE_DIR}/{file_name}.p'
samples = evaluate(RUN_ID=45, load_ckpt="GRPO_2/checkpoint_200", num_samples=100, only_evaluate=True, samples_file=samples_path, single=True)

## 3. Sample conditioned on $N_{\rm flux}, |W_0|$ or $1/g_s$

Here we evaluate the samples from the models that have additional been conditioned on $N_{\rm flux}, |W_0|$ or $1/g_s$. Remember, if you changed anything in the sampling notebook, change here accordingly.

### $N_{\rm flux}$

In [ ]:
file_name="n_flux_sample"
samples_path = f'{SAMPLE_DIR}/{file_name}.p'
samples_nflux = evaluate(RUN_ID=38, load_ckpt="GRPO_8/checkpoint_400", num_samples=100, only_evaluate=True, samples_file=samples_path, nflux=80)

### $|W_0|$

In [ ]:
file_name="w0_sample"
samples_path = f'{SAMPLE_DIR}/{file_name}.p'
samples_w0 = evaluate(RUN_ID=39, load_ckpt="GRPO_2/checkpoint_400", num_samples=100, only_evaluate=True, samples_file=samples_path, w0=1.0)

### $1/g_s$

In [ ]:
file_name="gs_inv_sample"
samples_path = f'{SAMPLE_DIR}/{file_name}.p'
samples_gs_inv = evaluate(RUN_ID=46, load_ckpt="GRPO_5/checkpoint_400", num_samples=100, only_evaluate=True, samples_file=samples_path, gs_inv=3.0)

### Some plots

For the models that have be additionally prompted on $N_{\rm flux}, |W_0|$ or $1/g_s$, we can generate some interesting plots. Namely, the distribution of valid sampled values around the prompted value. First, we need to define a plotting function.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib_inline.backend_inline import set_matplotlib_formats

# High-resolution notebook output
set_matplotlib_formats('retina')

def plot(cond, prompt, valid_samples):
  dfs = []
  max_val = []
  min_val = []
  for entry in valid_samples:
    max_val.append(np.max(valid_samples[entry]["dist"]))
    min_val.append(np.min(valid_samples[entry]["dist"]))
    dfs.append(pd.DataFrame({cond: valid_samples[entry]["dist"]}))

  sns.set_theme()
  plt.rcParams.update({
    "text.usetex": False,        # Use built-in MathText
    "mathtext.fontset": "cm",    # Computer Modern (LaTeX-style font)
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "font.size": 6,
    "axes.labelsize": 6,
    "axes.titlesize": 6,
    "legend.fontsize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "figure.dpi": 150,
})
  fig, ax = plt.subplots(4, 5, figsize=(8,8), sharey='row')
  fig.subplots_adjust(
    left=0.08,
    right=0.98,
    bottom=0.10,
    top=0.92,
    hspace=0.5,
    wspace=0.12
)
  all_hodge_pairs = np.array(
      [
          [[2, 120], [2, 128], [2, 132], [2, 144], [2, 272]],
          [[3, 99], [3, 107], [3, 115], [3, 119], [3, 243]],
          [[4, 98], [4, 118], [4, 122], [4, 130], [4, 142]],
          [[5, 89], [5, 149], [5, 153], [5, 165], [5, 213]],
      ]
  )
  for row in range(4):
      for col in range(5):
          if cond == r"$N_{\rm flux}$":
            discrete = True
            bins=None
          else:
            discrete = False
            bins = np.arange(0.,6.1, 0.1)
          sns.histplot(data=dfs[5*row+col], x=cond, ax=ax[row, col], discrete=discrete, bins=bins)
          if cond == r"$N_{\rm flux}$": 
            ax[row, col].axvline(prompt, color="red", linestyle='--')
          else:
            ax[row, col].axvline(prompt-0.05, color="red", linestyle='--')
            ax[row, col].set_xlim([min_val[5*row+col]-1, max_val[5*row+col]+1])
          hp = all_hodge_pairs[row, col]
          ax[row, col].set_title(r"$(h^{1,2}, h^{1,1})$ = " + f"({hp[0]}, {hp[1]})")


  plt.show()

Then we call it for each condition separately. Remember to change the prompt value if you it was changed in the sampling notebook.

#### $N_{\rm flux}$

In [ ]:
plot(r"$N_{\rm flux}$", prompt=80, valid_samples=samples_nflux)

#### $|W_0|$

In [ ]:
plot(r"$|W_0|$", prompt=1.0, valid_samples=samples_w0)

#### $1/g_s$

In [ ]:
plot(r"$1/g_s$", prompt=3.0, valid_samples=samples_gs_inv)